# Strawberry Disease Detection: Training (Google Colab)

Trains a YOLO object detector on the Roboflow **Strawberry Diseases Detection** dataset (project-raggj, 9 classes, MIT license).

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Create a free account at https://roboflow.com, copy your API key from *Settings → API Keys*
3. In Colab, open the 🔑 **Secrets** panel (left sidebar), add a secret named `ROBOFLOW_API_KEY`, paste the key, and enable notebook access

Everything is saved to Google Drive (`MyDrive/hydro_tower/`), so results survive if Colab disconnects.

## 1. Settings
Change `MODEL` to run the thesis comparison experiments (`yolo11n.pt`, `yolo11s.pt`, `yolov8n.pt`, ...).

In [ ]:
MODEL = "yolo11n.pt"      # n = fastest on Jetson; s = more accurate, slower
EPOCHS = 100
IMGSZ = 640
BATCH = 16

RF_WORKSPACE = "project-raggj"
RF_PROJECT = "strawberry-diseases-detection-abknq"
RF_VERSION = None          # None = use the latest dataset version

DRIVE_DIR = "/content/drive/MyDrive/hydro_tower"
RUN_NAME = MODEL.replace(".pt", "")

## 2. Setup

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_DIR, exist_ok=True)

## 3. Download the dataset

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)

if RF_VERSION is None:
    available = sorted(int(str(v.version).split("/")[-1]) for v in project.versions())
    print("Available dataset versions:", available)
    RF_VERSION = available[-1]

ds = project.version(RF_VERSION).download("yolov8", location="/content/dataset")
print("Downloaded version", RF_VERSION, "to", ds.location)

Roboflow's `data.yaml` uses relative paths that often break Ultralytics, so rewrite them as absolute paths and check the splits.

In [ ]:
import yaml, glob

yaml_path = f"{ds.location}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)

data["path"] = ds.location
data["train"] = "train/images"
data["val"] = "valid/images"
if os.path.isdir(f"{ds.location}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)

with open(yaml_path, "w") as f:
    yaml.safe_dump(data, f)

print("Classes:", data["names"])
for split in ["train", "valid", "test"]:
    print(f"{split:6s}", len(glob.glob(f"{ds.location}/{split}/images/*")), "images")

## 4. Train
Starts from COCO-pretrained weights (transfer learning). Takes roughly 1–2 h on a T4. `patience` stops early if validation mAP stops improving.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=20,
    project=f"{DRIVE_DIR}/runs",
    name=RUN_NAME,
    exist_ok=True,
)

**If Colab disconnected mid-training:** rerun sections 1–3, then run this cell instead of the one above to continue from the last checkpoint.

In [ ]:
# model = YOLO(f"{DRIVE_DIR}/runs/{RUN_NAME}/weights/last.pt")
# model.train(resume=True)

## 5. Evaluate
Training curves and confusion matrix (these plots go in the thesis).

In [ ]:
from IPython.display import Image, display

run_dir = f"{DRIVE_DIR}/runs/{RUN_NAME}"
for name in ["results.png", "confusion_matrix_normalized.png"]:
    display(Image(filename=f"{run_dir}/{name}", width=900))

Final scores on the **test** split, which the model never saw during training. Report these numbers, not the validation ones.

In [ ]:
best = YOLO(f"{run_dir}/weights/best.pt")
split = "test" if "test" in data else "val"
metrics = best.val(data=yaml_path, split=split, project=f"{DRIVE_DIR}/runs", name=f"{RUN_NAME}_{split}", exist_ok=True)

print(f"\n{split} mAP50    = {metrics.box.map50:.3f}")
print(f"{split} mAP50-95 = {metrics.box.map:.3f}")
print("\nPer-class mAP50-95:")
for i, cls_map in zip(metrics.box.ap_class_index, metrics.box.maps[metrics.box.ap_class_index]):
    print(f"  {best.names[i]:25s} {cls_map:.3f}")

Visual check on a few test images.

In [ ]:
import random, cv2

test_imgs = glob.glob(f"{ds.location}/{'test' if split == 'test' else 'valid'}/images/*")
for r in best.predict(random.sample(test_imgs, 6), conf=0.4, verbose=False):
    display(Image(data=cv2.imencode(".jpg", r.plot())[1].tobytes(), width=500))

## 6. Export for the Jetson
Exports ONNX as a portable backup. The TensorRT `.engine` must be built **on the Jetson itself** from `best.pt` (engines are tied to the GPU and TensorRT version).

In [ ]:
best.export(format="onnx", imgsz=IMGSZ)
print("Copy these to the Jetson:")
!ls -lh {run_dir}/weights/